# 05 – Sinkhorn Re-ranking (Phase 6)

Builds on Phase 4 best: **WiSE-FT α=0.3 (SigLIP + LogReg C=100)** — val R-prec = 0.9287, LB = 0.9511.

No GPU needed — all on cached features.

In [ ]:
import sys, pathlib
sys.path.insert(0, "/kaggle/input/datasets/somyajitsaha/food-retrieval-src")
import numpy as np
import pandas as pd
from src.config import seed_everything
seed_everything(42)

DATA_DIR = pathlib.Path("/kaggle/input/competitions/match-the-dish-to-its-rizz")
OUT_DIR  = pathlib.Path("/kaggle/working")
(OUT_DIR / "submissions").mkdir(exist_ok=True)
print("DATA_DIR exists:", DATA_DIR.exists())

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
from src.data import load_csvs, parse_val_ground_truth
from src.text_matching import build_surface_form_dict, TextMatcher

dfs          = load_csvs(DATA_DIR)
train_df     = dfs["train"]
val_gallery  = dfs["val_gallery"]
val_queries  = dfs["val_queries"]
val_gt_df    = dfs["val_gt"]
test_gallery = dfs["test_gallery"]
test_queries = dfs["test_queries"]
meta         = dfs["category_metadata"]
val_gt       = parse_val_ground_truth(val_gt_df)

cat_id_list = meta["category_id"].tolist()
cat_id2idx  = {c: i for i, c in enumerate(cat_id_list)}   # string keys
K           = len(cat_id_list)
matcher     = TextMatcher(build_surface_form_dict(meta), meta_df=meta)
print(f"K={K}")

In [ ]:
# ── Reproduce Phase 4 best: WiSE-FT α=0.3 on SigLIP features ─────────────────
from src.features import load_features, load_text_features
from src.ensemble import temperature_scale, blend_zeroshot_finetuned
from sklearn.linear_model import LogisticRegression

feat_dir = OUT_DIR / "features" / "siglip_so400m"

X_train, train_ids = load_features(feat_dir, "train", gallery_df=train_df)
X_val,   val_ids   = load_features(feat_dir, "val",   gallery_df=val_gallery)
X_test,  test_ids  = load_features(feat_dir, "test",  gallery_df=test_gallery)
T_proto, _         = load_text_features(feat_dir, tag="text")

id2cat = dict(zip(train_df["image_id"], train_df["category_id"]))
y_enc  = np.array([cat_id2idx[id2cat[i]] for i in train_ids])

clf = LogisticRegression(C=100.0, max_iter=2000, solver="lbfgs",
                          multi_class="multinomial", random_state=42, n_jobs=-1)
clf.fit(X_train, y_enc)
print(f"LogReg train acc: {clf.score(X_train, y_enc):.4f}")

def row_norm(S):
    return (S - S.mean(axis=1, keepdims=True)) / (S.std(axis=1, keepdims=True) + 1e-9)

def make_wiseft(X, T, alpha=0.3):
    S_zs = X @ T.T
    S_lr = clf.decision_function(X)
    return temperature_scale(
        blend_zeroshot_finetuned(row_norm(S_zs), row_norm(S_lr), alpha=alpha),
        T=1.0, to_prob=True)

S_val  = make_wiseft(X_val,  T_proto)
S_test = make_wiseft(X_test, T_proto)
print(f"Score matrices: val={S_val.shape}  test={S_test.shape}")

In [ ]:
# ── Helper ────────────────────────────────────────────────────────────────────
from src.metrics import evaluate, print_metrics

def score_to_preds(S, gallery_ids, query_df, top=20):
    pred_dict = {}
    for _, row in query_df.iterrows():
        cat_id, _, _ = matcher.predict(str(row["description"]))
        cat_idx = cat_id2idx.get(cat_id, 0)
        order   = np.argsort(-S[:, cat_idx], kind="stable")
        pred_dict[str(row["query_id"])] = [gallery_ids[j] for j in order[:top]]
    return pred_dict

# Reference
preds   = score_to_preds(S_val, val_ids, val_queries)
metrics = evaluate(preds, val_gt)
print("WiSE-FT α=0.3 (no re-ranking):")
print_metrics(metrics)
ablation = [{"config": "WiSE-FT (no rerank)", **metrics}]

In [ ]:
# ── Sinkhorn strength sweep ───────────────────────────────────────────────────
from src.rerank import sinkhorn_balance

print("Sinkhorn strength sweep:")
best_str, best_rp = 0.0, -1.0

for strength in np.arange(0.0, 1.05, 0.1):
    S_rr  = sinkhorn_balance(S_val, n_iter=20, strength=float(strength))
    preds = score_to_preds(S_rr, val_ids, val_queries)
    rp    = evaluate(preds, val_gt)["r_precision"]
    mark  = " ←" if rp > best_rp else ""
    print(f"  strength={strength:.1f}  R-prec={rp:.4f}{mark}")
    if rp > best_rp:
        best_rp, best_str = rp, float(strength)

print(f"\nBest Sinkhorn: strength={best_str:.1f}  R-prec={best_rp:.4f}")

In [ ]:
# Evaluate best Sinkhorn
S_sink_val  = sinkhorn_balance(S_val,  n_iter=20, strength=best_str)
S_sink_test = sinkhorn_balance(S_test, n_iter=20, strength=best_str)
preds   = score_to_preds(S_sink_val, val_ids, val_queries)
metrics = evaluate(preds, val_gt)
ablation.append({"config": f"+ Sinkhorn str={best_str:.1f}", **metrics})
print(f"WiSE-FT + Sinkhorn (str={best_str:.1f}):")
print_metrics(metrics)

In [ ]:
# ── kNN smoothing sweep ───────────────────────────────────────────────────────
from src.rerank import knn_smooth

print("kNN alpha sweep (k=5):")
best_knn_alpha, best_knn_rp = 0.0, -1.0

for alpha in [0.1, 0.2, 0.3, 0.4, 0.5]:
    S_knn = knn_smooth(S_val, X_val, k=5, alpha=alpha)
    preds = score_to_preds(S_knn, val_ids, val_queries)
    rp    = evaluate(preds, val_gt)["r_precision"]
    mark  = " ←" if rp > best_knn_rp else ""
    print(f"  alpha={alpha:.1f}  R-prec={rp:.4f}{mark}")
    if rp > best_knn_rp:
        best_knn_rp, best_knn_alpha = rp, alpha

print(f"\nBest kNN: alpha={best_knn_alpha:.1f}  R-prec={best_knn_rp:.4f}")

In [ ]:
# kNN alone
S_knn_val  = knn_smooth(S_val,  X_val,  k=5, alpha=best_knn_alpha)
S_knn_test = knn_smooth(S_test, X_test, k=5, alpha=best_knn_alpha)
preds   = score_to_preds(S_knn_val, val_ids, val_queries)
metrics = evaluate(preds, val_gt)
ablation.append({"config": f"+ kNN alpha={best_knn_alpha:.1f}", **metrics})
print(f"WiSE-FT + kNN (α={best_knn_alpha:.1f}):")
print_metrics(metrics)

# Sinkhorn + kNN combined
S_both_val  = knn_smooth(S_sink_val,  X_val,  k=5, alpha=best_knn_alpha)
S_both_test = knn_smooth(S_sink_test, X_test, k=5, alpha=best_knn_alpha)
preds   = score_to_preds(S_both_val, val_ids, val_queries)
metrics = evaluate(preds, val_gt)
ablation.append({"config": "+ Sinkhorn + kNN", **metrics})
print(f"WiSE-FT + Sinkhorn + kNN:")
print_metrics(metrics)

In [ ]:
# ── Ablation table ────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("ABLATION TABLE")
print("="*70)
abl_df = pd.DataFrame(ablation)
print(abl_df[["config","r_precision","map@20","p@5","hit@1","mrr@20"]].to_string(index=False))

best_row    = max(ablation, key=lambda x: x["r_precision"])
best_config = best_row["config"]
print(f"\n→ Best: {best_config}  R-prec={best_row['r_precision']:.4f}")

In [ ]:
# ── Build final submission ────────────────────────────────────────────────────
from src.submit import build_submission, validate_submission
import datetime

if "Sinkhorn + kNN" in best_config:
    S_final, tag = S_both_test,  "wiseft_sink_knn"
elif "Sinkhorn" in best_config:
    S_final, tag = S_sink_test,  f"wiseft_sink{best_str:.1f}"
elif "kNN" in best_config:
    S_final, tag = S_knn_test,   f"wiseft_knn{best_knn_alpha:.1f}"
else:
    S_final, tag = S_test,       "wiseft_a0.3_norerank"

test_query_ids, test_cat_indices = [], []
for _, row in test_queries.iterrows():
    cat_id, _, _ = matcher.predict(str(row["description"]))
    test_query_ids.append(str(row["query_id"]))
    test_cat_indices.append(cat_id2idx.get(cat_id, 0))

sub_df   = build_submission(S_final, test_ids, test_query_ids, test_cat_indices, top=20)
is_valid = validate_submission(sub_df, test_gallery)

ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
sub_path = OUT_DIR / "submissions" / f"{ts}_{tag}.csv"
sub_df.to_csv(sub_path, index=False)
sub_df.to_csv(OUT_DIR / "submission.csv", index=False)
print(f"Saved: {sub_path.name}")

import subprocess, sys
r = subprocess.run(
    [sys.executable, str(DATA_DIR/"validate_submission.py"),
     str(OUT_DIR/"submission.csv"), "--data-dir", str(DATA_DIR)],
    capture_output=True, text=True)
print(r.stdout)
if r.stderr: print(r.stderr)
print("Validator exit code:", r.returncode)
print(f"\nFinal val R-prec = {best_row['r_precision']:.4f}  config = {best_config}")